In [ ]:

import cv2
import numpy as np

def Webcame(cameras : int):
    if type(cameras) == int and cameras:
        return cv2.VideoCapture(cameras)
camera = Webcame(0)
if not camera.isOpened():
    raise RuntimeError("Failed to Access the Webcam")

clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8, 8)
)

COLOR_PALETTE = {
    "Porcelain": (242, 219, 202),
    "Ivory": (237, 205, 180),
    "Beige": (226, 190, 160),
    "Peach": (220, 165, 135),
    "Tan": (190, 135, 100),
    "Caramel": (160, 105, 72),
    "Brown": (120, 75, 50),
    "Deep Brown": (75, 45, 32)
}

PALETTE_LAB = {}

for name, rgb in COLOR_PALETTE.items():

    color = np.uint8([[rgb]])
    lab = cv2.cvtColor(color,cv2.COLOR_RGB2LAB)[0][0]
    PALETTE_LAB[name] = lab


def get_color_name(bgr):

    color = np.uint8([[bgr]])

    lab = cv2.cvtColor(
        color,
        cv2.COLOR_BGR2LAB
    )[0][0]

    distances = {
        name: np.linalg.norm(lab - palette_color)
        for name, palette_color in PALETTE_LAB.items()
    }

    return min(distances, key=distances.get)


while True:

    success, frame = camera.read()

    if not success:
        break

    frame = cv2.flip(frame, 1)

    hsv = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2HSV
    )

    h, s, v = cv2.split(hsv)

    v = clahe.apply(v)

    hsv = cv2.merge((h, s, v))

    lower_hsv = np.array([0, 15, 30])
    upper_hsv = np.array([30, 200, 255])

    hsv_mask = cv2.inRange(
        hsv,
        lower_hsv,
        upper_hsv
    )

    ycrcb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2YCrCb
    )

    lower_ycrcb = np.array([0, 125, 65])
    upper_ycrcb = np.array([255, 180, 140])

    ycrcb_mask = cv2.inRange(
        ycrcb,
        lower_ycrcb,
        upper_ycrcb
    )

    mask = cv2.bitwise_and(
        hsv_mask,
        ycrcb_mask
    )

    kernel = np.ones((5, 5), np.uint8)

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for contour in contours:

        area = cv2.contourArea(contour)

        if area < 5000:
            continue

        hand_mask = np.zeros(
            frame.shape[:2],
            dtype=np.uint8
        )

        cv2.drawContours(
            hand_mask,
            [contour],
            -1,
            255,
            -1
        )

        hand_mask = cv2.bitwise_and(
            hand_mask,
            mask
        )

        pixels = frame[hand_mask > 0]

        if len(pixels) == 0:
            continue

        median_bgr = np.median(
            pixels,
            axis=0
        ).astype(np.uint8)

        color_name = get_color_name(
            median_bgr
        )

        b, g, r = map(int, median_bgr)

        color_hex = f"#{r:02X}{g:02X}{b:02X}"

        x, y, w, h = cv2.boundingRect(contour)

        label = f"Hands | Skin Color : {color_name}"

        cv2.drawContours(
            frame,
            [contour],
            -1,
            (0, 255, 0),
            2
        )

        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (255, 0, 0),
            2
        )

        cv2.putText(
            frame,
            label,
            (x, max(y - 12, 25)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 0),
            2
        )

        cv2.rectangle(
            frame,
            (x, y + h + 5),
            (x + 35, y + h + 40),
            (b, g, r),
            -1
        )

    cv2.imshow("Hand Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

camera.release()
cv2.destroyAllWindows()